In [0]:
%fs ls abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/

In [0]:
# Reading/loading the parquet file 
df_invoices_101_200 = spark.read.parquet("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet")

# Creating two copies of the parquet file in two different folders.
df_invoices_101_200.write.mode("overwrite").parquet("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v1")
df_invoices_101_200.write.mode("overwrite").parquet("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v2")

In [0]:
%sql
CONVERT TO DELTA
PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v1`

In [0]:
from delta.tables import DeltaTable

DeltaTable.convertToDelta(
    spark,
    "PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v2`"
)

In [0]:
# Reading/loading converted parquet to delta file 
df_invoices_101_200 = spark.read.format("delta").load("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v1")
display(df_invoices_101_200)

In [0]:
delta_table = DeltaTable.forPath(spark, "abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_101_200_v1")
display(delta_table.history())

In [0]:
# Reading/loading the source csv file 
df_Gold_reserves_tonnes = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Gold_reserves_tonnes.csv")
# Correcting the column name for delta table compatibility
df_Gold_reserves_tonnes = df_Gold_reserves_tonnes.withColumnRenamed("Average gold reserves", "Average_gold_reserves")
display(df_Gold_reserves_tonnes)

# Converting csv file to delta file
df_Gold_reserves_tonnes.write.mode("overwrite").format("delta").save("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Gold_reserves_tonnes_delta")

In [0]:
# Reading/loading converted csv to delta file 
df_Gold_reserves_tonnes = spark.read.format("delta").load("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/Gold_reserves_tonnes_delta")
display(df_Gold_reserves_tonnes)

In [0]:
%sql
-- Till now whatever delta tables we have created in prevous notebooks were managed tables.
-- Now we will create an external table in delta format.
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_ext
  USING DELTA
  LOCATION 'abfss://dbr-external-tables-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices_ext' AS
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ext;

DESCRIBE EXTENDED delta_catalog.delta_db.invoices_ext;